In [1]:
import pandas as pd
import numpy as np

# Load dataset dengan encoding latin-1 karena ada karakter khusus
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# Membuang kolom ekstra yang kosong jika ada (berdasarkan struktur csv)
df = df.iloc[:, :2]
df.columns = ['v1', 'v2']

# Mapping label 'spam' dan 'ham' ke bentuk numerik
new_labels = {
    'spam': 1,
    'ham': 0
}
df['target'] = df['v1'].map(new_labels)

# Menentukan fitur (X) dan target (y)
X = df['v2']
y = df['target']

df.head()

,v1,v2,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data 80:20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inisialisasi CountVectorizer dengan stop_words
vectorizer_cv = CountVectorizer(stop_words='english')

# Transformasi data teks menjadi matriks fitur
X_train_cv = vectorizer_cv.fit_transform(X_train)
X_test_cv = vectorizer_cv.transform(X_test)

print(f"Dimensi fitur training (CountVectorizer): {X_train_cv.shape}")

Dimensi fitur training (CountVectorizer): (4457, 7505)


In [3]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Inisialisasi dan latih model Multinomial Naive Bayes
nb_cv = MultinomialNB()
nb_cv.fit(X_train_cv, y_train)

# Prediksi data uji
y_pred_cv = nb_cv.predict(X_test_cv)

# Evaluasi
acc_cv = accuracy_score(y_test, y_pred_cv)
print("=== EVALUASI MODEL: Multinomial NB + CountVectorizer ===")
print(f"Akurasi: {acc_cv * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_cv))

=== EVALUASI MODEL: Multinomial NB + CountVectorizer ===
Akurasi: 98.57%

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.96      0.93      0.95       150

    accuracy                           0.99      1115
   macro avg       0.97      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TfidfVectorizer dengan stop_words
vectorizer_tfidf = TfidfVectorizer(stop_words='english')

# Transformasi data teks menjadi matriks TF-IDF
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

print(f"Dimensi fitur training (TF-IDF): {X_train_tfidf.shape}")

Dimensi fitur training (TF-IDF): (4457, 7505)


In [5]:
# Inisialisasi dan latih model Multinomial Naive Bayes dengan TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

# Prediksi data uji
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

# Evaluasi
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
print("=== EVALUASI MODEL: Multinomial NB + TF-IDF ===")
print(f"Akurasi: {acc_tfidf * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf))

=== EVALUASI MODEL: Multinomial NB + TF-IDF ===
Akurasi: 96.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.75      0.85       150

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [6]:
# Ringkasan Perbandingan Performa
comparison = pd.DataFrame({
    'Metode Ekstraksi Fitur': ['CountVectorizer (stop_words)', 'TF-IDF (stop_words)'],
    'Akurasi (%)': [acc_cv * 100, acc_tfidf * 100]
})

print("=== TABEL PERBANDINGAN AKURASI ===")
print(comparison.to_string(index=False))

print("\n=== KESIMPULAN ===")
if acc_cv > acc_tfidf:
    print("Fitur CountVectorizer dengan stop_words memberikan hasil akurasi yang lebih baik pada kasus spam.csv ini.")
elif acc_tfidf > acc_cv:
    print("Fitur TF-IDF dengan stop_words memberikan hasil akurasi yang lebih baik karena mampu meredam bobot kata-kata yang terlalu sering muncul.")
else:
    print("Kedua metode memberikan tingkat akurasi yang relatif setara pada dataset ini.")

=== TABEL PERBANDINGAN AKURASI ===
      Metode Ekstraksi Fitur  Akurasi (%)
CountVectorizer (stop_words)    98.565022
         TF-IDF (stop_words)    96.591928

=== KESIMPULAN ===
Fitur CountVectorizer dengan stop_words memberikan hasil akurasi yang lebih baik pada kasus spam.csv ini.
